# Summary of all runs
Reads every finished run in `results/*.json` and builds the tables and figures for the report.
Figures are saved to `results/figures/`.

In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("results/figures", exist_ok=True)

def method(c):
    if c["server_optimizer"]=="adam":
        return "FedAdam"
    return {"sgd":"FedAvg", "sgd_momentum":"FedAvg + momentum", "fedprox":"FedProx"}.get(c["client_optimizer"], c["client_optimizer"])

rows=[]
for path in sorted(glob.glob("results/*.json")):
    r=json.load(open(path))
    if not r["finished"]:
        continue
    c=r["config"]
    rows.append({**c, "method":method(c),
                 "final_acc":np.mean(r["test_accuracy"][-5:]),     # mean of the last 5 rounds
                 "drift":np.mean(r["client_drift"]),
                 "dissimilarity":np.median(r["update_dissimilarity"]),
                 "sec_per_round":np.mean(r["seconds_per_round"]),
                 "acc_curve":r["test_accuracy"], "drift_curve":r["client_drift"]})
df=pd.DataFrame(rows)
print(len(df), "finished runs")

## 1. Tuning (stage = tune, seed 11)
Every configuration ran at alpha=0.5 and alpha=0.1. Copy the best row per method into `best_settings()` in
`run_experiments.sh`: the alpha=0.5 best into "mild" (used for alpha 0.5 and 100), the alpha=0.1 best into "strong"
(used for alpha 0.1 and the 1-class split). Accuracy near chance (1.6% FEMNIST, 10% CIFAR) means the run diverged.

In [ ]:
tune=df[df.stage=="tune"]
if len(tune):
    for setting, regime in [("alpha0.5","mild"), ("alpha0.1","strong")]:
        t=tune[tune.setting==setting].sort_values(["dataset","method","final_acc"], ascending=[True,True,False])
        print(f"Best configuration per method at {setting}  ->  \"{regime}\" in best_settings()")
        display(t.groupby(["dataset","method"]).head(1)[["dataset","method","client_lr","server_lr","mu","final_acc"]])
    print("All tuning runs:")
    display((tune.pivot_table(index=["dataset","method","client_lr","server_lr","mu"], columns="setting", values="final_acc")*100).round(1))

## 2. Main result: final test accuracy (mean ± std over seeds)
Columns: `alphaX` = Dirichlet split with that alpha, `c1` = pathological split with 1 class per client.

In [ ]:
grid=df[df.stage=="grid"]
table=grid.groupby(["dataset","method","setting"]).final_acc.agg(["mean","std","count"])
pretty=((table["mean"]*100).round(1).astype(str)+" ± "+(table["std"]*100).round(1).astype(str)).unstack("setting")
display(pretty)
print(table["count"].unstack("setting"))    # number of seeds per cell
with open("results/figures/accuracy_table.tex","w") as f:
    f.write(pretty.to_latex())

In [ ]:
dirichlet=grid[grid.partition=="dirichlet"]      # accuracy vs alpha only makes sense for the Dirichlet split
for dataset, g in dirichlet.groupby("dataset"):
    fig,axs=plt.subplots(1,2, figsize=(11,4))
    for m, gm in g.groupby("method"):
        acc=gm.groupby("alpha").final_acc.agg(["mean","std"])
        axs[0].errorbar(acc.index, acc["mean"]*100, yerr=acc["std"]*100, marker="o", capsize=3, label=m)
        dis=gm.groupby("alpha").dissimilarity.agg(["mean","std"])
        axs[1].errorbar(dis.index, dis["mean"], yerr=dis["std"], marker="o", capsize=3, label=m)
    axs[0].set_ylabel("Final test accuracy (%)")
    axs[1].set_ylabel("Client update dissimilarity")
    for ax in axs:
        ax.set_xscale("log")
        ax.set_xlabel("Dirichlet alpha  (left = more heterogeneous)")
        ax.legend()
    fig.suptitle(dataset)
    plt.tight_layout()
    plt.savefig(f"results/figures/{dataset}_accuracy_and_dissimilarity_vs_alpha.png", dpi=200)
    plt.show()

## 3. Learning curves and client drift (mean over seeds)

In [ ]:
for (dataset, setting), g in grid.groupby(["dataset","setting"]):
    fig,axs=plt.subplots(1,2, figsize=(11,4))
    for m, gm in g.groupby("method"):
        acc=np.mean(np.stack(gm.acc_curve.values), axis=0)
        drift=np.mean(np.stack(gm.drift_curve.values), axis=0)
        axs[0].plot(acc*100, label=m)
        axs[1].plot(drift, label=m)
    axs[0].set_ylabel("Test accuracy (%)")
    axs[1].set_ylabel("Client drift (mean ||w_k - w||)")
    for ax in axs:
        ax.set_xlabel("Round")
        ax.legend()
    fig.suptitle(f"{dataset}, {setting}")
    plt.tight_layout()
    plt.savefig(f"results/figures/{dataset}_{setting}_curves.png", dpi=200)
    plt.show()

## 4. Communication cost: rounds needed to reach a target accuracy
Adjust the targets after looking at the curves (a target that FedAvg reaches at alpha=0.5 is a good choice).

In [ ]:
targets={"femnist":0.75, "cifar":0.55}

def rounds_to(curve, target):
    hits=np.nonzero(np.array(curve)>=target)[0]
    return hits[0]+1 if len(hits) else np.nan       # NaN = never reached

grid=grid.assign(rounds_to_target=[rounds_to(c, targets[d]) for c,d in zip(grid.acc_curve, grid.dataset)])
display(grid.groupby(["dataset","method","setting"]).rounds_to_target.agg(["mean","std"]).round(1).unstack("setting"))